# MobileViT for CIFAR-10 Classification

This notebook implements MobileViT, a lightweight hybrid architecture combining CNNs and Transformers, for CIFAR-10 image classification.

## Setup and Installation

In [ ]:
# Install required packages
!pip install torch torchvision tqdm matplotlib -q

In [ ]:
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import time
from tqdm import tqdm
import matplotlib.pyplot as plt
import numpy as np

# Check if CUDA is available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Using device: {device}')
if torch.cuda.is_available():
    print(f'GPU: {torch.cuda.get_device_name(0)}')
    print(f'CUDA Version: {torch.version.cuda}')

## Upload MobileViT Implementation

Upload the `mobilevit.py` file to this Colab session using the file uploader below.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload mobilevit.py

from mobilevit import mobilevit_xxs, mobilevit_xs, mobilevit_s
print("MobileViT module loaded successfully!")

## Load CIFAR-10 Dataset

In [ ]:
def get_cifar10_loaders(batch_size=128, num_workers=2):
    """Create CIFAR-10 data loaders with augmentation."""
    
    # Data augmentation for training
    train_transform = transforms.Compose([
        transforms.RandomCrop(32, padding=4),
        transforms.RandomHorizontalFlip(),
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    
    # No augmentation for validation
    val_transform = transforms.Compose([
        transforms.ToTensor(),
        transforms.Normalize((0.4914, 0.4822, 0.4465), (0.2023, 0.1994, 0.2010)),
    ])
    
    # Load datasets
    train_dataset = datasets.CIFAR10(
        root='./data',
        train=True,
        download=True,
        transform=train_transform,
    )
    
    val_dataset = datasets.CIFAR10(
        root='./data',
        train=False,
        download=True,
        transform=val_transform,
    )
    
    # Create data loaders
    train_loader = DataLoader(
        train_dataset,
        batch_size=batch_size,
        shuffle=True,
        num_workers=num_workers,
        pin_memory=True,
    )
    
    val_loader = DataLoader(
        val_dataset,
        batch_size=batch_size,
        shuffle=False,
        num_workers=num_workers,
        pin_memory=True,
    )
    
    return train_loader, val_loader

# Create data loaders
batch_size = 128
train_loader, val_loader = get_cifar10_loaders(batch_size=batch_size)

print(f'Training samples: {len(train_loader.dataset)}')
print(f'Validation samples: {len(val_loader.dataset)}')
print(f'Number of classes: 10')
print(f'Classes: {train_loader.dataset.classes}')

In [ ]:
# Create model (choose variant: xxs, xs, or s)
model_variant = 'xxs'  # Options: 'xxs', 'xs', 's'

if model_variant == 'xxs':
    model = mobilevit_xxs(num_classes=10)
elif model_variant == 'xs':
    model = mobilevit_xs(num_classes=10)
elif model_variant == 's':
    model = mobilevit_s(num_classes=10)

model = model.to(device)

# Count parameters
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

num_params = count_parameters(model)
print(f'Model: MobileViT-{model_variant.upper()}')
print(f'Number of parameters: {num_params:,}')

# Test forward pass with a dummy input
dummy_input = torch.randn(1, 3, 32, 32).to(device)
with torch.no_grad():
    output = model(dummy_input)
    print(f'Output shape: {output.shape}')

## Training Functions

In [ ]:
def train_epoch(model, train_loader, criterion, optimizer, device, epoch):
    """Train for one epoch."""
    model.train()
    running_loss = 0.0
    correct = 0
    total = 0
    
    pbar = tqdm(train_loader, desc=f'Epoch {epoch+1} [Train]')
    for inputs, targets in pbar:
        inputs, targets = inputs.to(device), targets.to(device)
        
        # Forward pass
        optimizer.zero_grad()
        outputs = model(inputs)
        loss = criterion(outputs, targets)
        
        # Backward pass
        loss.backward()
        optimizer.step()
        
        # Statistics
        running_loss += loss.item()
        _, predicted = outputs.max(1)
        total += targets.size(0)
        correct += predicted.eq(targets).sum().item()
        
        # Update progress bar
        pbar.set_postfix({
            'loss': f'{running_loss/(pbar.n+1):.4f}',
            'acc': f'{100.*correct/total:.2f}%'
        })
    
    epoch_loss = running_loss / len(train_loader)
    epoch_acc = 100. * correct / total
    
    return epoch_loss, epoch_acc


def validate(model, val_loader, criterion, device):
    """Validate the model."""
    model.eval()
    running_loss = 0.0
    correct = 0
    total = 0
    
    with torch.no_grad():
        pbar = tqdm(val_loader, desc='[Val]')
        for inputs, targets in pbar:
            inputs, targets = inputs.to(device), targets.to(device)
            
            outputs = model(inputs)
            loss = criterion(outputs, targets)
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            total += targets.size(0)
            correct += predicted.eq(targets).sum().item()
            
            pbar.set_postfix({
                'loss': f'{running_loss/(pbar.n+1):.4f}',
                'acc': f'{100.*correct/total:.2f}%'
            })
    
    epoch_loss = running_loss / len(val_loader)
    epoch_acc = 100. * correct / total
    
    return epoch_loss, epoch_acc

In [ ]:
# Training hyperparameters
num_epochs = 100
learning_rate = 0.001
weight_decay = 0.01

# Loss and optimizer
criterion = nn.CrossEntropyLoss()
optimizer = optim.AdamW(
    model.parameters(),
    lr=learning_rate,
    weight_decay=weight_decay,
)

# Learning rate scheduler
scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer,
    T_max=num_epochs,
    eta_min=1e-6,
)

# Training history
history = {
    'train_loss': [],
    'train_acc': [],
    'val_loss': [],
    'val_acc': [],
}

best_val_acc = 0.0

print(f'Starting training for {num_epochs} epochs...')
print('=' * 60)

start_time = time.time()

for epoch in range(num_epochs):
    # Train
    train_loss, train_acc = train_epoch(
        model, train_loader, criterion, optimizer, device, epoch
    )
    
    # Validate
    val_loss, val_acc = validate(model, val_loader, criterion, device)
    
    # Update learning rate
    scheduler.step()
    
    # Save history
    history['train_loss'].append(train_loss)
    history['train_acc'].append(train_acc)
    history['val_loss'].append(val_loss)
    history['val_acc'].append(val_acc)
    
    # Save best model
    if val_acc > best_val_acc:
        best_val_acc = val_acc
        torch.save({
            'epoch': epoch,
            'model_state_dict': model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'val_acc': val_acc,
        }, 'best_model.pth')
    
    # Print epoch summary
    print(f'\nEpoch {epoch+1}/{num_epochs}:')
    print(f'  Train Loss: {train_loss:.4f}, Train Acc: {train_acc:.2f}%')
    print(f'  Val Loss: {val_loss:.4f}, Val Acc: {val_acc:.2f}%')
    print(f'  Best Val Acc: {best_val_acc:.2f}%')
    print(f'  LR: {scheduler.get_last_lr()[0]:.6f}')
    print('-' * 60)

total_time = time.time() - start_time
print(f'\nTraining completed in {total_time/60:.2f} minutes')
print(f'Best validation accuracy: {best_val_acc:.2f}%')

In [ ]:
# Plot training history
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(15, 5))

# Loss plot
ax1.plot(history['train_loss'], label='Train Loss')
ax1.plot(history['val_loss'], label='Val Loss')
ax1.set_xlabel('Epoch')
ax1.set_ylabel('Loss')
ax1.set_title('Training and Validation Loss')
ax1.legend()
ax1.grid(True)

# Accuracy plot
ax2.plot(history['train_acc'], label='Train Acc')
ax2.plot(history['val_acc'], label='Val Acc')
ax2.set_xlabel('Epoch')
ax2.set_ylabel('Accuracy (%)')
ax2.set_title('Training and Validation Accuracy')
ax2.legend()
ax2.grid(True)

plt.tight_layout()
plt.show()

## Evaluate on Test Set

In [ ]:
# Load best model
checkpoint = torch.load('best_model.pth')
model.load_state_dict(checkpoint['model_state_dict'])
print(f"Loaded model from epoch {checkpoint['epoch']+1} with val acc: {checkpoint['val_acc']:.2f}%")

# Final evaluation
test_loss, test_acc = validate(model, val_loader, criterion, device)
print(f'\nFinal Test Accuracy: {test_acc:.2f}%')

## Test Inference on Sample Images

In [ ]:
# Get some sample images
dataiter = iter(val_loader)
images, labels = next(dataiter)
images, labels = images.to(device), labels.to(device)

# Make predictions
model.eval()
with torch.no_grad():
    outputs = model(images[:8])
    _, predicted = outputs.max(1)

# Display results
class_names = train_loader.dataset.classes

fig, axes = plt.subplots(2, 4, figsize=(12, 6))
axes = axes.flatten()

for idx in range(8):
    img = images[idx].cpu()
    # Denormalize
    img = img * torch.tensor([0.2023, 0.1994, 0.2010]).view(3, 1, 1) + torch.tensor([0.4914, 0.4822, 0.4465]).view(3, 1, 1)
    img = torch.clamp(img, 0, 1)
    
    axes[idx].imshow(img.permute(1, 2, 0))
    axes[idx].set_title(f'True: {class_names[labels[idx]]}\nPred: {class_names[predicted[idx]]}',
                        color='green' if labels[idx] == predicted[idx] else 'red')
    axes[idx].axis('off')

plt.tight_layout()
plt.show()